model_14_9_4.xlsx', model_24_9_15.xlsx', _model_24_9_5.xlsx', model_24_9_8.xlsx', model_26_8_14.xlsx', model_31_9_8.xlsx', model_35_7_20.xlsx', model_36_6_0.xlsx', model_8_4_3.xlsx', model_9_9_2.xlsx'


In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score

def redes(file_names,file_names_1000):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    Z_pred_sum = np.sum(Z_pred_total, axis=1).reshape(-1, 1)
    df = pd.read_excel("1000_model/1000_model_14_9_4.xlsx")
    Z = df['Z'].values.reshape(-1, 1)
    mse_sup = np.mean((Z - Z_pred_sum/10) ** 2)

    df_list1 = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds1 = [df['Z_pred'].values.reshape(-1, 1) for df in df_list1]
    Z_pred_total1 = np.hstack(z_preds1)
    Z_pred_sum1 = np.sum(Z_pred_total1, axis=1).reshape(-1, 1)
    df1 = pd.read_excel("25_model/25_model_14_9_4.xlsx")
    Z1 = df1['Z'].values.reshape(-1, 1)

    mse_sup1 = np.mean((Z1 - Z_pred_sum1/10) ** 2)
    r2_sup_1000 = r2_score(Z, Z_pred_sum/10)
    r2_sup_25 = r2_score(Z1, Z_pred_sum1/10)
    
    print(f"r2_25: {r2_sup_25}")
    print(f"r2_1000: {r2_sup_1000}")
    print(f"mse_1000: {mse_sup}")
    print(f"mse_25: {mse_sup1}")
    return mse_sup

# usar:
file_names_1000 = [
    "1000_model/1000_model_14_9_4.xlsx",
    "1000_model/1000_model_24_9_15.xlsx",
    "1000_model/1000_model_24_9_8.xlsx",
    "1000_model/1000_model_26_8_14.xlsx",
    "1000_model/1000_model_31_9_8.xlsx",
    "1000_model/1000_model_35_7_20.xlsx",
    "1000_model/1000_model_36_6_0.xlsx",
    "1000_model/1000_model_8_4_3.xlsx",
    "1000_model/1000_model_9_9_2.xlsx",
    "1000_model/1000_model_24_9_5.xlsx"
]

file_names = [
    "25_model/25_model_14_9_4.xlsx",
    "25_model/25_model_24_9_15.xlsx",
    "25_model/25_model_26_8_14.xlsx",
    "25_model/25_model_31_9_8.xlsx",
    "25_model/25_model_35_7_20.xlsx",
    "25_model/25_model_36_6_0.xlsx",
    "25_model/25_model_8_4_3.xlsx",
    "25_model/25_model_24_9_8.xlsx",
    "25_model/25_model_9_9_2.xlsx",
    "25_model/25_model_24_9_5.xlsx"
]

result= redes(file_names, file_names_1000)



r2_25: 0.999068151854873
r2_1000: 0.9952217818172371
mse_1000: 1803.7773639930022
mse_25: 351.7726747854746


In [9]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt



def redes(file_names):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds = np.hstack([df['Z_pred'].values.reshape(-1,1) for df in df_list])
    dfZ = pd.read_excel("25_model/25_model_14_9_4.xlsx")
    Z = dfZ['Z'].values.reshape(-1, 1)
    N = Z.shape[0]
    return Z, z_preds, N

def pesos(Z, z_preds, N):  
    
    best_erro = np.inf
    best_w = None
    patience_counter = 0
    a = np.random.uniform(0, 1, z_preds.shape[1])

    for epoch in range(1, n_epocas+1):
        w = np.exp(a) / np.sum(np.exp(a)) # softmax
        yhat = np.dot(z_preds, w)
        residuo = Z.flatten() - yhat
        mse = np.mean(residuo**2)
        mse_pond = mse + lambda_reg * np.sum(a**2)
        # gradiente dmse/dw
        gmse = (-2.0 / N) * z_preds.T.dot(residuo)
        # gradiente completo
        s = np.dot(gmse, w)
        grad_a = w * (gmse - s) + 2.0 * lambda_reg * a
        # atualização
        a = a - lr * grad_a

        # ---- EARLY STOPPING ----
        if mse_pond < best_erro - min_delta:
            best_erro = mse_pond
            best_w = w.copy()
            patience_counter = 0  # reset
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"\nEarly stopping ativado na época {epoch} — perda não melhorou por {patience} épocas.")
            break
        # -------------------------

        if epoch % 100 == 0 or epoch == 1:
            r2 = 1 - np.sum((Z - yhat.reshape(-1,1))**2) / np.sum((Z - Z.mean())**2)
            #print(f"epoch {epoch:4d} loss_25={mse_pond:.6f} mse_25={mse:.6f} R2_25={r2:.4f} weights={w}")

    return best_erro, best_w

def mse(file_names, file_names_1000, best_w):
    df_list = [pd.read_excel(arquivo) for arquivo in file_names_1000]
    z_preds = [df['Z_pred'].values.reshape(-1, 1) for df in df_list]
    Z_pred_total = np.hstack(z_preds)
    yhat_final = Z_pred_total * best_w      
    Z_pred_sum = np.sum(yhat_final, axis=1).reshape(-1, 1)
    df1 = pd.read_excel("1000_model/1000_model_14_9_4.xlsx")
    Z1 = df1['Z'].values.reshape(-1, 1)
    df_list2 = [pd.read_excel(arquivo) for arquivo in file_names]
    z_preds2 = [df['Z_pred'].values.reshape(-1, 1) for df in df_list2]
    Z_pred_total2 = np.hstack(z_preds2)
    yhat_final2 = Z_pred_total2 * best_w      
    Z_pred_sum2 = np.sum(yhat_final2, axis=1).reshape(-1, 1)
    df2 = pd.read_excel("25_model/25_model_14_9_4.xlsx")
    Z2 = df2['Z'].values.reshape(-1, 1)

    mse_sup_1000 = np.mean((Z1 - Z_pred_sum) ** 2)
    mse_25 = np.mean((Z2 - Z_pred_sum2) ** 2)
    r2_sup_1000 = r2_score(Z1, Z_pred_sum)
    r2_sup_25 = r2_score(Z2, Z_pred_sum2)

    print(f"MSE conjunto = {mse_sup_1000}")
    print(f"MSE 25 dados = {mse_25}")
    print(f"R² conjunto = {r2_sup_1000}")
    print(f"R² 25 dados = {r2_sup_25}")

    return mse_sup_1000, r2_sup_1000, mse_25


file_names_1000 =  ["1000_model/1000_model_14_9_4.xlsx", "1000_model/1000_model_24_9_15.xlsx",
              "1000_model/1000_model_24_9_8.xlsx", "1000_model/1000_model_26_8_14.xlsx", 
              "1000_model/1000_model_31_9_8.xlsx", "1000_model/1000_model_35_7_20.xlsx", 
              "1000_model/1000_model_36_6_0.xlsx","1000_model/1000_model_8_4_3.xlsx",
              "1000_model/1000_model_9_9_2.xlsx", "1000_model/1000_model_24_9_5.xlsx"]

file_names = [
    "25_model/25_model_14_9_4.xlsx",
    "25_model/25_model_24_9_15.xlsx",
    "25_model/25_model_26_8_14.xlsx",
    "25_model/25_model_31_9_8.xlsx",
    "25_model/25_model_35_7_20.xlsx",
    "25_model/25_model_36_6_0.xlsx",
    "25_model/25_model_8_4_3.xlsx",
    "25_model/25_model_24_9_8.xlsx",
    "25_model/25_model_9_9_2.xlsx",
    "25_model/25_model_24_9_5.xlsx"
]





In [11]:


lambda_reg = 1e-5        
lr = 1e-3                 
n_epocas = 500000
patience=500
min_delta=1e-6

Z, z_preds, N= redes(file_names)
best_erro, best_w = pesos(Z, z_preds, N)

print("best erro =", best_erro)
print("best pesos =", best_w)
mse_1000 =mse(file_names, file_names_1000, best_w)


best erro = 17.55868693769901
best pesos = [3.07052894e-01 6.06492064e-06 7.55010244e-06 4.56742174e-06
 1.72349898e-05 1.37346650e-01 6.46955967e-06 6.03461268e-06
 5.55546496e-01 6.03855436e-06]
MSE conjunto = 2269.97470122667
MSE 25 dados = 17.556275046950255
R² conjunto = 0.9939868219835055
R² 25 dados = 0.9999534933111339


In [36]:
lambda_reg = 1e-4        
lr = 5e-4                 
n_epocas = 50000
patience=500
min_delta=1e-8

best_erro1, best_w_1 = pesos(Z, z_preds, N)

print("best erro =", best_erro1)
print("best pesos =", best_w_1)
mse_1000 =mse_1000(file_names_1000, best_w)
mse_25 =mse_25(file_names_25, best_w)

epoch    1 loss_25=378.749058 mse_25=378.748812 R2_25=0.9990 weights=[0.09122757 0.11653446 0.09727773 0.09439604 0.1179606  0.09419122
 0.10347175 0.06856818 0.07593523 0.14043722]
epoch  100 loss_25=45.138544 mse_25=45.137172 R2_25=0.9999 weights=[0.55758868 0.02425797 0.02344971 0.01944074 0.04428401 0.17124256
 0.02288097 0.01857966 0.09207394 0.02620177]
epoch  200 loss_25=34.649135 mse_25=34.647263 R2_25=0.9999 weights=[0.67271021 0.01478034 0.01459743 0.0116219  0.02781777 0.14258498
 0.0140832  0.01166309 0.07433534 0.01580575]
epoch  300 loss_25=32.780590 mse_25=32.778479 R2_25=0.9999 weights=[0.69038    0.01210672 0.01211969 0.00935649 0.02331897 0.14467703
 0.01160306 0.00968526 0.07386388 0.0128889 ]
epoch  400 loss_25=31.676156 mse_25=31.673880 R2_25=0.9999 weights=[0.68262722 0.01081686 0.01094517 0.00823621 0.02125864 0.15659839
 0.01041508 0.00873849 0.07888434 0.01147959]
epoch  500 loss_25=30.716137 mse_25=30.713721 R2_25=0.9999 weights=[0.66469646 0.00997575 0.010185

TypeError: 'tuple' object is not callable

In [83]:
lambda_reg = 5e-5
lr = 1e-6
n_epocas = 10000
patience = 300
min_delta = 1e-6

best_erro1, best_w_1 = pesos(Z, z_preds, N)

print("best erro =", best_erro1)
print("best pesos =", best_w_1)
mse_1000 =mse(file_names_1000, best_w_1)

epoch    1 loss_25=415.681273 mse_25=415.681164 R2_25=0.9989 weights=[0.07726325 0.127078   0.09232102 0.14384867 0.13596557 0.07098083
 0.07517435 0.06967735 0.07195284 0.13573814]
epoch  100 loss_25=414.078770 mse_25=414.078662 R2_25=0.9989 weights=[0.07780016 0.12672707 0.09220061 0.14316527 0.13639437 0.07137472
 0.07506622 0.06959238 0.07234742 0.13533177]
epoch  200 loss_25=412.459088 mse_25=412.458981 R2_25=0.9989 weights=[0.07834664 0.12637166 0.09207719 0.14247824 0.13682268 0.0717742
 0.07495584 0.06950542 0.07274753 0.13492058]
epoch  300 loss_25=410.838453 mse_25=410.838347 R2_25=0.9989 weights=[0.07889731 0.12601532 0.09195197 0.14179449 0.13724605 0.07217529
 0.07484433 0.06941733 0.0731492  0.13450871]
epoch  400 loss_25=409.216912 mse_25=409.216806 R2_25=0.9989 weights=[0.07945219 0.12565809 0.09182497 0.141114   0.13766439 0.07257797
 0.07473169 0.06932811 0.07355239 0.13409619]
epoch  500 loss_25=407.594509 mse_25=407.594404 R2_25=0.9989 weights=[0.08001133 0.1253    

In [29]:
lambda_reg = 5e-5
lr = 1e-6
n_epocas = 10000
patience = 300
min_delta = 1e-6

best_erro1, best_w_1 = pesos(Z, z_preds, N)

print("best erro =", best_erro1)
print("best pesos =", best_w_1)
mse_1000 =mse(file_names_1000, best_w,"1000_model/1000_model_9_9_2.xlsx")
mse_25 =mse(file_names_25, best_w,"25_model/25_model_24_9_5.xlsx" )

epoch    1 loss_25=372.508160 mse_25=372.507987 R2_25=0.9990 weights=[0.06478836 0.07980329 0.11801416 0.10687373 0.08469339 0.14955501
 0.15321627 0.1027033  0.07667447 0.06367803]
epoch  100 loss_25=370.495660 mse_25=370.495488 R2_25=0.9990 weights=[0.06509969 0.07961658 0.11762741 0.10644412 0.08484758 0.15091811
 0.15246472 0.10239179 0.07702981 0.06356019]
epoch  200 loss_25=368.461451 mse_25=368.461278 R2_25=0.9990 weights=[0.06541364 0.07942608 0.11723468 0.10600972 0.0849993  0.15230719
 0.15170692 0.10207522 0.07738777 0.06343947]
epoch  300 loss_25=366.426036 mse_25=366.425864 R2_25=0.9990 weights=[0.06572704 0.07923368 0.1168399  0.10557489 0.08514692 0.15370858
 0.1509505  0.10175675 0.07774468 0.06331706]
epoch  400 loss_25=364.389605 mse_25=364.389432 R2_25=0.9990 weights=[0.06603982 0.07903938 0.11644311 0.10513964 0.0852904  0.15512229
 0.15019549 0.10143641 0.07810049 0.06319297]
epoch  500 loss_25=362.352344 mse_25=362.352172 R2_25=0.9990 weights=[0.06635193 0.0788432

epoch 5400 loss_25=265.633455 mse_25=265.633258 R2_25=0.9993 weights=[0.07968052 0.06742701 0.09501074 0.08346683 0.08667654 0.24092348
 0.11483763 0.08380686 0.0929915  0.0551789 ]
epoch 5500 loss_25=263.805904 mse_25=263.805706 R2_25=0.9993 weights=[0.0798908  0.06716925 0.09457082 0.08304809 0.08658892 0.24290638
 0.11419055 0.08343898 0.09320485 0.05499136]
epoch 5600 loss_25=261.987247 mse_25=261.987048 R2_25=0.9993 weights=[0.08009792 0.06691103 0.09413127 0.08263044 0.08649721 0.2448974
 0.11354629 0.08307119 0.09341409 0.05480318]
epoch 5700 loss_25=260.177633 mse_25=260.177433 R2_25=0.9993 weights=[0.08030183 0.06665238 0.09369213 0.08221389 0.08640146 0.24689636
 0.11290486 0.08270353 0.09361919 0.05461437]
epoch 5800 loss_25=258.377208 mse_25=258.377007 R2_25=0.9993 weights=[0.08050251 0.06639335 0.09325346 0.08179849 0.08630171 0.24890306
 0.11226629 0.08233604 0.09382013 0.05442497]
epoch 5900 loss_25=256.586116 mse_25=256.585914 R2_25=0.9993 weights=[0.08069993 0.06613397

In [81]:
lambda_reg = 5e-5
lr = 1e-6
n_epocas = 10000
patience = 300
min_delta = 1e-6

best_erro1, best_w_1 = pesos(Z, z_preds, N)

print("best erro =", best_erro1)
print("best pesos =", best_w_1)
mse_1000 =mse(file_names_1000, best_w_1)

epoch    1 loss_25=437.749937 mse_25=437.749768 R2_25=0.9988 weights=[0.0704344  0.1308465  0.06204195 0.10272692 0.13880825 0.06433166
 0.13223351 0.10851482 0.06055757 0.12950442]
epoch  100 loss_25=436.268955 mse_25=436.268786 R2_25=0.9988 weights=[0.07091093 0.1304888  0.06201264 0.1024187  0.1393438  0.06467987
 0.13184832 0.10828157 0.06086042 0.12915493]
epoch  200 loss_25=434.771230 mse_25=434.771062 R2_25=0.9988 weights=[0.07139617 0.130127   0.06198203 0.10210735 0.13988117 0.06503328
 0.13145904 0.10804507 0.0611675  0.12880139]
epoch  300 loss_25=433.271756 mse_25=433.271589 R2_25=0.9989 weights=[0.07188536 0.12976473 0.06195039 0.10179599 0.14041483 0.06538836
 0.13106959 0.10780768 0.06147573 0.12844735]
epoch  400 loss_25=431.770577 mse_25=431.770411 R2_25=0.9989 weights=[0.07237854 0.129402   0.06191774 0.10148463 0.14094467 0.06574512
 0.13067997 0.1075694  0.06178512 0.12809281]
epoch  500 loss_25=430.267735 mse_25=430.267570 R2_25=0.9989 weights=[0.07287574 0.1290388